In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
import pickle
import tqdm
from IPython.display import clear_output
%matplotlib inline

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

# desktop version
data_folder = "G:\Meu Drive\mestrado_final_files\data\\"
models_folder = "G:\Meu Drive\mestrado_final_files\models\\"

# google colab version
# data_folder = "drive/MyDrive/mestrado_final_files/data/"
# models_folder = "drive/MyDrive/mestrado_final_files/models/"

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [4]:
def fit_pipeline_model(CONTEXT_SIZE, EMBEDDING_LAYER_SIZE, NUM_HIDDEN_NEURONS, constant_value, fitted_models = [], root_path = "", tp_model = "simplevars", device = "cpu"):
    filename_model_token = "model_token_{}_context{}_emb{}_hiddenlayers{}_numneurons{}_multiplicador{}_parameters.pth"
    filename_model_time = "model_time_{}_context{}_emb{}_hiddenlayers{}_numneurons{}_multiplicador{}_parameters.pth"

    str_model_token = filename_model_token.format(tp_model, CONTEXT_SIZE, EMBEDDING_LAYER_SIZE, len(NUM_HIDDEN_NEURONS), NUM_HIDDEN_NEURONS[0], constant_value)
    str_model_time = filename_model_time.format(tp_model, CONTEXT_SIZE, EMBEDDING_LAYER_SIZE, len(NUM_HIDDEN_NEURONS), NUM_HIDDEN_NEURONS[0], constant_value)

    if str_model_token in fitted_models:
        print("Já ajustado. Faça outra configuração")
        return []

    results_dict = {"context": CONTEXT_SIZE, "embedding_size": EMBEDDING_LAYER_SIZE, "hidden_neurons": NUM_HIDDEN_NEURONS,
                    "constant_multiplicator": constant_value, "filename_token_model": str_model_token, "filename_time_model": str_model_time}

    print("Treinando modelo de previsao proximo token")
    model_token = NGramModel_ExtraInfo(context_size = CONTEXT_SIZE, vocab_size = VOCAB_SIZE, embedding_dim = EMBEDDING_LAYER_SIZE,
                                       numerical_input_dim = len(index_covariables), num_hidden_neurons = NUM_HIDDEN_NEURONS,
                                       num_layers = len(NUM_HIDDEN_NEURONS), activation = nn.GELU, dropout_rate = 0.3)

    model_token = model_token.to(device)
    model_token, trnloss, devloss = train_num_ngram_model(model = model_token,
                                                          train_token_data = selected_games_tkn_trn['token_features'][:, -CONTEXT_SIZE:],
                                                          train_numerical_data = constant_value * selected_games_tkn_trn['numerical_features'][:, index_covariables],
                                                          train_targets = selected_games_tkn_trn['target_tokens'],
                                                          batch_size = BATCH_SIZE,
                                                          max_steps = TRAIN_STEPS,
                                                          lr = 0.002,
                                                          val_token_data = selected_games_tkn_dev['token_features'][:, -CONTEXT_SIZE:],
                                                          val_numerical_data = constant_value * selected_games_tkn_dev['numerical_features'][:, index_covariables],
                                                          val_targets = selected_games_tkn_dev['target_tokens'],
                                                          params_optimizer = {'factor': 0.5, 'threshold': 0, 'patience': 0})

    print("\nSalvando pesos do modelo para prever proximo token")
    torch.save(model_token.state_dict(), root_path + str_model_token)

    results_dict["token_model_train_loss"] = trnloss
    results_dict["token_model_dev_loss"] = devloss

    total = 0
    for name, param in model_token.named_parameters():
        total += param.numel()

    results_dict["token_model_total_parameters"] = str(total)

    print(f"\nQuantidade de parametros do modelo de token: {total}")

    output = model_token(selected_games_tkn_trn['token_features'][:, -CONTEXT_SIZE:], constant_value * selected_games_tkn_trn['numerical_features'][:, index_covariables])
    baseline_loss = F.cross_entropy(output, selected_games_tkn_trn['target_tokens'])

    results_dict["token_model_traindata_loss"] = float(baseline_loss)

    print(f"\nFunção de perda para os dados de treino no modelo de token: {float(baseline_loss)}")

    print("\nCalculando a importancia das variaveis de input:")

    df_importance = permutation_importance_ngram(model = model_token, target = selected_games_tkn_dev['target_tokens'],
                                                 tokens = selected_games_tkn_dev['token_features'][:, -CONTEXT_SIZE:],
                                                 numerical_features = constant_value * selected_games_tkn_dev['numerical_features'][:, index_covariables],
                                                 names_numerical_features = covariables_model, num_repeats = 10)

    print(df_importance)

    results_dict["token_model_importance_variable"] = df_importance

    df_first_layer_weights = get_descriptive_values_weights(model_token, emb_layer_size = EMBEDDING_LAYER_SIZE, context_size = CONTEXT_SIZE, covariables = covariables_model)

    results_dict["token_model_weights_describe"] = df_first_layer_weights

    print("\n\nTreinando modelo de previsao proximo tempo")

    time_model = SpentTimeModel(context_size = CONTEXT_SIZE, vocab_size = VOCAB_SIZE, time_size = TIME_SIZE,
                                embedding_dim = EMBEDDING_LAYER_SIZE, numerical_input_dim = len(covariables_model),
                                num_hidden_neurons = NUM_HIDDEN_NEURONS, num_layers = len(NUM_HIDDEN_NEURONS),
                                activation_function = nn.GELU, dropout_rate = 0.3)

    time_model = time_model.to(device)
    time_model, time_trnlosses, time_devlosses = train_spent_time_model(
        model = time_model,
        train_token_data = selected_games_time_trn['token_features'][:, -CONTEXT_SIZE:],
        train_numerical_data = constant_value * selected_games_time_trn['numerical_features'][:, index_covariables],
        train_targets = selected_games_time_trn['target_time'],
        batch_size = BATCH_SIZE,
        max_steps = TRAIN_STEPS,
        lr =  0.002,
        val_token_data = selected_games_time_dev['token_features'][:, -CONTEXT_SIZE:],
        val_numerical_data =  constant_value * selected_games_time_dev['numerical_features'][:, index_covariables],
        val_targets =  selected_games_time_dev['target_time'],
        params_optimizer = {'factor': 0.5, 'threshold': 1e-5, 'patience': 0})

    print("\nSalvando pesos do modelo para prever proximo tempo")
    torch.save(time_model.state_dict(), root_path + str_model_time)

    results_dict["time_model_train_loss"] = time_trnlosses
    results_dict["time_model_dev_loss"] = time_devlosses

    total = 0
    for name, param in time_model.named_parameters():
        total += param.numel()

    results_dict["time_model_total_parameters"] = str(total)

    print(f"\nQuantidade de parametros do modelo de token: {total}")

    output = time_model(selected_games_time_trn['token_features'][:, -CONTEXT_SIZE:], constant_value * selected_games_time_trn['numerical_features'][:, index_covariables])
    baseline_loss = F.cross_entropy(output, selected_games_time_trn['target_time'])

    results_dict["time_model_traindata_loss"] = float(baseline_loss)

    print(f"\nFunção de perda para os dados de treino no modelo de token: {float(baseline_loss)}")

    print("\nCalculando a importancia das variaveis de input:")

    df_importance = permutation_importance_ngram(model = time_model, target = selected_games_time_dev['target_time'],
                                                 tokens = selected_games_time_dev['token_features'][:, -CONTEXT_SIZE:],
                                                 numerical_features = constant_value * selected_games_time_dev['numerical_features'][:, index_covariables],
                                                 names_numerical_features = covariables_model, num_repeats = 10)

    print(df_importance)

    results_dict["time_model_importance_variable"] = df_importance

    df_first_layer_weights = get_descriptive_values_weights(time_model, emb_layer_size = EMBEDDING_LAYER_SIZE, context_size = CONTEXT_SIZE, covariables = covariables_model)

    results_dict["time_model_weights_describe"] = df_first_layer_weights

    return results_dict

In [5]:
with open(data_folder + 'integer_to_token.pkl', 'rb') as f:
    integer_to_token = pickle.load(f)

In [6]:
with open(data_folder + 'data_token_train.pkl', 'rb') as f:
    tkn_trn = pickle.load(f)

with open(data_folder + 'data_token_validation.pkl', 'rb') as f:
    tkn_dev = pickle.load(f)


In [7]:
with open(data_folder + 'data_time_train.pkl', 'rb') as f:
    time_trn = pickle.load(f)

with open(data_folder+ 'data_time_validation.pkl', 'rb') as f:
    time_dev = pickle.load(f)

In [8]:
selected_games_tkn_trn = {}
for k in ['token_features', 'numerical_features', 'target_tokens']:
    selected_games_tkn_trn[k] = [item for key, item in tkn_trn[k].items()]
    selected_games_tkn_trn[k] = [sublist for list_ in selected_games_tkn_trn[k] for sublist in list_]
    if k == 'numerical_features':
        selected_games_tkn_trn[k] = torch.tensor(selected_games_tkn_trn[k]).float()
    else:
        selected_games_tkn_trn[k] = torch.tensor(selected_games_tkn_trn[k])
    selected_games_tkn_trn[k] = selected_games_tkn_trn[k].to(device)

selected_games_time_trn = {}
for k in ['token_features', 'numerical_features', 'target_time']:
    selected_games_time_trn[k] = [item for key, item in time_trn[k].items()]
    selected_games_time_trn[k] = [sublist for list_ in selected_games_time_trn[k] for sublist in list_]
    if k == 'numerical_features':
        selected_games_time_trn[k] = torch.tensor(selected_games_time_trn[k]).float()
    else:
        selected_games_time_trn[k] = torch.tensor(selected_games_time_trn[k])
    selected_games_time_trn[k] = selected_games_time_trn[k].to(device)

In [9]:
selected_games_tkn_dev = {}
for k in ['token_features', 'numerical_features', 'target_tokens']:
    selected_games_tkn_dev[k] = [item for key, item in tkn_dev[k].items()]
    selected_games_tkn_dev[k] = [sublist for list_ in selected_games_tkn_dev[k] for sublist in list_]
    if k == 'numerical_features':
        selected_games_tkn_dev[k] = torch.tensor(selected_games_tkn_dev[k]).float()
    else:
        selected_games_tkn_dev[k] = torch.tensor(selected_games_tkn_dev[k])
    selected_games_tkn_dev[k] = selected_games_tkn_dev[k].to(device)

selected_games_time_dev = {}
for k in ['token_features', 'numerical_features', 'target_time']:
    selected_games_time_dev[k] = [item for key, item in time_dev[k].items()]
    selected_games_time_dev[k] = [sublist for list_ in selected_games_time_dev[k] for sublist in list_]
    if k == 'numerical_features':
        selected_games_time_dev[k] = torch.tensor(selected_games_time_dev[k]).float()
    else:
        selected_games_time_dev[k] = torch.tensor(selected_games_time_dev[k])
    selected_games_time_dev[k] = selected_games_time_dev[k].to(device)


In [10]:
del tkn_trn, tkn_dev, time_trn, time_dev

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 60)

In [12]:
df_games_nba = pd.read_csv(data_folder + "games_nba.csv", sep = ';')\
[['GAME_ID', 'HOME_TEAM_ID', 'AWAY_TEAM_ID', 'HOME_TEAM', 'AWAY_TEAM', 'GAME_DATE', 'SEASON']]

In [32]:
full_covariables = ['period1', 'period2', 'period3', 'period4', 'period_overtime', 'time', 'limhfouls', 'limvfouls', 'dif', 'dif_20', 'ind_close_3',
                    'ind_close_5', 'ind_close_8', 'ind_blowout_15', 'ind_last_2_minutes', 'ind_last_minute', 'ind_last_20_seconds', 'ind_last_10_seconds', 
                    'lead', 'home_lead', 'away_lead', 'ind_last_2_minutes_game', 'ind_last_minute_game', 'ind_last_20_seconds_game', 
                    'ind_last_10_seconds_game', 'ind_clutch_2_last_minutes', 'ind_clutch_2_last_minutes_home', 'ind_clutch_2_last_minutes_away',
                    'ind_clutch_last_minute', 'ind_clutch_last_minute_home', 'ind_clutch_last_minute_away', 'ind_clutch_last_20_seconds',
                    'ind_clutch_last_20_seconds_home', 'ind_clutch_last_20_seconds_away', 'ind_clutch_last_10_seconds', 'ind_clutch_last_10_seconds_home',
                    'ind_clutch_last_10_seconds_away', 'ind_blowout_last_minute', 'ball_possession', 'home_FGM', 'away_FGM', 'home_FGA', 'away_FGA', 
                    'home_FGP', 'away_FGP', 'home_3FGM', 'away_3FGM', 'home_3FGA', 'away_3FGA', 'home_3FGP', 'away_3FGP', 'home_FTM', 'away_FTM', 
                    'home_FTA', 'away_FTA', 'home_FTP', 'away_FTP', 'home_RebOff', 'away_RebOff', 'home_RebDef', 'away_RebDef', 'home_RebTot', 
                    'away_RebTot', 'home_AST', 'away_AST', 'home_STL', 'away_STL', 'home_BLK', 'away_BLK', 'home_TO', 'away_TO', 'home_FP', 
                    'away_FP', 'home_points', 'away_points', 'home_PMP', 'away_PMP', 'home_outcome', 'away_outcome', 'playoff_game']

covariables_model = ['period1', 'period2', 'period3', 'period4', 'period_overtime', 'time', 'limhfouls', 'limvfouls', 'dif_20', 'ind_close_3',
                    'ind_close_5', 'ind_close_8', 'ind_blowout_15', 'ind_last_2_minutes', 'ind_last_minute', 'ind_last_20_seconds', 'ind_last_10_seconds', 
                    'home_lead', 'away_lead', 'ind_last_2_minutes_game', 'ind_last_minute_game', 'ind_last_20_seconds_game', 
                    'ind_last_10_seconds_game', 'ind_clutch_2_last_minutes', 'ind_clutch_2_last_minutes_home', 'ind_clutch_2_last_minutes_away',
                    'ind_clutch_last_minute', 'ind_clutch_last_minute_home', 'ind_clutch_last_minute_away', 'ind_clutch_last_20_seconds',
                    'ind_clutch_last_20_seconds_home', 'ind_clutch_last_20_seconds_away', 'ind_clutch_last_10_seconds', 'ind_clutch_last_10_seconds_home',
                    'ind_clutch_last_10_seconds_away', 'ind_blowout_last_minute', 'ball_possession']

# covariables_model = ['period1', 'period2', 'period3', 'period4', 'period_overtime', 'time', 'limhfouls', 'limvfouls', 'dif', 'ind_last_minute',
#                      'ind_last_20_seconds', 'ind_last_10_seconds', 'lead', 'ball_possession', 'home_3FGM', 'away_3FGM', 'home_3FGA', 'away_3FGA',
#                      'home_FTM', 'away_FTM', 'home_FTA', 'away_FTA', 'home_RebOff', 'away_RebOff', 'home_RebDef', 'away_RebDef', 'home_AST', 'away_AST',
#                      'home_STL', 'away_STL', 'home_BLK', 'away_BLK', 'home_TO', 'away_TO', 'home_FP', 'away_FP', 'home_points', 'away_points',
#                      'playoff_game']

# covariables_model = ['period1', 'period2', 'period3', 'period4', 'period_overtime', 'time', 'limhfouls', 'limvfouls', 'dif', 'ind_last_minute',
#                      'ind_last_20_seconds', 'ind_last_10_seconds', 'lead', 'ball_possession']

index_covariables = [i for i, var in enumerate(full_covariables) if var in covariables_model]

In [33]:
BATCH_SIZE = 1024
TRAIN_STEPS = 50001
VOCAB_SIZE = 601
TIME_SIZE = 301

run_models = 1

In [34]:
%run 01_functions_training.ipynb
#%run drive/MyDrive/mestrado_final_files/notebooks/01_functions_training.ipynb

In [35]:
model_token1 = NGramModel_ExtraInfo(context_size = 5, vocab_size = VOCAB_SIZE, embedding_dim = 64,
                                    numerical_input_dim = len(index_covariables), num_hidden_neurons = [100, 100, 100],
                                    num_layers = 3, activation = nn.GELU, dropout_rate = 0.3)

model_token1 = model_token1.to(device)
model_token1, trnloss1, devloss1 = train_num_ngram_model(model = model_token1,
                                                        train_token_data = selected_games_tkn_trn['token_features'][:, -5:],
                                                        train_numerical_data = 1 * selected_games_tkn_trn['numerical_features'][:, index_covariables],
                                                        train_targets = selected_games_tkn_trn['target_tokens'],
                                                        batch_size = BATCH_SIZE,
                                                        max_steps = TRAIN_STEPS,
                                                        lr = 0.002,
                                                        val_token_data = selected_games_tkn_dev['token_features'][:, -5:],
                                                        val_numerical_data = 1 * selected_games_tkn_dev['numerical_features'][:, index_covariables],
                                                        val_targets = selected_games_tkn_dev['target_tokens'],
                                                        params_optimizer = {'factor': 0.5, 'threshold': 0, 'patience': 0})

In [36]:
torch.save(model_token1.state_dict(), models_folder + "model_ngram_tokens_new_variables_parameters.pth")


In [37]:
time_model1 = SpentTimeModel(context_size = 5, vocab_size = VOCAB_SIZE, time_size = TIME_SIZE,
                            embedding_dim = 64, numerical_input_dim = len(covariables_model),
                            num_hidden_neurons = [100, 100, 100], num_layers = 3,
                            activation_function = nn.GELU, dropout_rate = 0.3)

time_model1, time_trnlosses1, time_devlosses1 = train_spent_time_model(
    model = time_model1,
    train_token_data = selected_games_time_trn['token_features'][:, -5:],
    train_numerical_data = selected_games_time_trn['numerical_features'][:, index_covariables],
    train_targets = selected_games_time_trn['target_time'],
    batch_size = BATCH_SIZE, max_steps = TRAIN_STEPS, lr =  0.002,
    val_token_data = selected_games_time_dev['token_features'][:, -5:],
    val_numerical_data =  selected_games_time_dev['numerical_features'][:, index_covariables],
    val_targets =  selected_games_time_dev['target_time'],
    params_optimizer = {'factor': 0.5, 'threshold': 1e-5, 'patience': 0})

In [38]:
torch.save(time_model1.state_dict(), models_folder + "model_ngram_time_new_variables_parameters.pth")


In [22]:
covariables_model2 = ['period1', 'period2', 'period3', 'period4', 'period_overtime', 'time', 'limhfouls', 'limvfouls', 'dif', 'ind_last_minute',
                     'ind_last_20_seconds', 'ind_last_10_seconds', 'lead', 'ball_possession']

index_covariables2 = [i for i, var in enumerate(full_covariables) if var in covariables_model2]

In [23]:
model_token2 = NGramModel_ExtraInfo(context_size = 5, vocab_size = VOCAB_SIZE, embedding_dim = 64,
                                    numerical_input_dim = len(index_covariables2), num_hidden_neurons = [100, 100, 100],
                                    num_layers = 3, activation = nn.GELU, dropout_rate = 0.3)

model_token2 = model_token2.to(device)
model_token2, trnloss2, devloss2 = train_num_ngram_model(model = model_token2,
                                                        train_token_data = selected_games_tkn_trn['token_features'][:, -5:],
                                                        train_numerical_data = 1 * selected_games_tkn_trn['numerical_features'][:, index_covariables2],
                                                        train_targets = selected_games_tkn_trn['target_tokens'],
                                                        batch_size = BATCH_SIZE,
                                                        max_steps = TRAIN_STEPS,
                                                        lr = 0.002,
                                                        val_token_data = selected_games_tkn_dev['token_features'][:, -5:],
                                                        val_numerical_data = 1 * selected_games_tkn_dev['numerical_features'][:, index_covariables2],
                                                        val_targets = selected_games_tkn_dev['target_tokens'],
                                                        params_optimizer = {'factor': 0.5, 'threshold': 0, 'patience': 0})

In [25]:
torch.save(model_token2.state_dict(), models_folder + "model_ngram_tokens_old_variables_parameters.pth")

In [30]:
time_model2 = SpentTimeModel(context_size = 5, vocab_size = VOCAB_SIZE, time_size = TIME_SIZE,
                            embedding_dim = 64, numerical_input_dim = len(index_covariables2),
                            num_hidden_neurons = [100, 100, 100], num_layers = 3,
                            activation_function = nn.GELU, dropout_rate = 0.3)

time_model2, time_trnlosses2, time_devlosses2 = train_spent_time_model(
    model = time_model2,
    train_token_data = selected_games_time_trn['token_features'][:, -5:],
    train_numerical_data = selected_games_time_trn['numerical_features'][:, index_covariables2],
    train_targets = selected_games_time_trn['target_time'],
    batch_size = BATCH_SIZE, max_steps = TRAIN_STEPS, lr =  0.002,
    val_token_data = selected_games_time_dev['token_features'][:, -5:],
    val_numerical_data =  selected_games_time_dev['numerical_features'][:, index_covariables2],
    val_targets =  selected_games_time_dev['target_time'],
    params_optimizer = {'factor': 0.5, 'threshold': 1e-5, 'patience': 0})

In [31]:
torch.save(time_model2.state_dict(), models_folder + "model_ngram_time_old_variables_parameters.pth")

In [16]:
with open(models_folder + "20251114_results_neural_networks_all_seasons_simple_vars.pkl", "rb") as f:
    list_results = pickle.load(f)

list_models_fitted = [item["filename_token_model"] for item in list_results]



In [33]:
sorted([item for item in list_models_fitted])

['model_token_simplevars_context10_emb16_hiddenlayers3_numneurons100_multiplicador10_parameters.pth',
 'model_token_simplevars_context10_emb16_hiddenlayers3_numneurons50_multiplicador1_parameters.pth',
 'model_token_simplevars_context10_emb32_hiddenlayers1_numneurons50_multiplicador5_parameters.pth',
 'model_token_simplevars_context10_emb32_hiddenlayers5_numneurons30_multiplicador5_parameters.pth',
 'model_token_simplevars_context10_emb64_hiddenlayers1_numneurons100_multiplicador1_parameters.pth',
 'model_token_simplevars_context10_emb64_hiddenlayers3_numneurons30_multiplicador10_parameters.pth',
 'model_token_simplevars_context10_emb8_hiddenlayers1_numneurons50_multiplicador3_parameters.pth',
 'model_token_simplevars_context10_emb8_hiddenlayers5_numneurons100_multiplicador3_parameters.pth',
 'model_token_simplevars_context1_emb16_hiddenlayers3_numneurons100_multiplicador3_parameters.pth',
 'model_token_simplevars_context1_emb16_hiddenlayers5_numneurons50_multiplicador1_parameters.pth'

In [28]:
len(list_models_fitted)

58

In [29]:
list_contexts = [5, 5, 10, 10, 10, 10]
list_embeddings = [8, 8, 8, 16, 32, 64]
list_num_hidden_neurons = [[30], [50, 50, 50], [100, 100, 100, 100, 100], [50, 50, 50], [50], [30, 30, 30]]
list_constant_values = [3, 1, 3, 1, 5, 10]

In [30]:
for ctx, emb, nhn, cv in zip(list_contexts, list_embeddings, list_num_hidden_neurons, list_constant_values):
    print(ctx, emb, nhn, cv)

    result = fit_pipeline_model(CONTEXT_SIZE = ctx, EMBEDDING_LAYER_SIZE = emb, NUM_HIDDEN_NEURONS = nhn, constant_value = cv,
                                fitted_models = list_models_fitted, root_path = models_folder, tp_model = "teamvars", device = "cuda")

    if len(result) > 0:
        list_results.append(result)
        with open(models_folder + "20251114_results_neural_networks_all_seasons_simple_vars.pkl", "wb") as f:
            pickle.dump(list_results, f)

        list_models_fitted = [item["filename_token_model"] for item in list_results]

    clear_output(wait = True)


10 64 [30, 30, 30] 10
Treinando modelo de previsao proximo token

Salvando pesos do modelo para prever proximo token

Quantidade de parametros do modelo de token: 79355

Função de perda para os dados de treino no modelo de token: 2.915104389190674

Calculando a importancia das variaveis de input:
Baseline model loss:  2.9173
                  index  count     mean     std      min      25%      50%  \
9               token_1   10.0  16.4297  0.0339  19.2941  19.3243  19.3476   
8               token_2   10.0   0.3820  0.0023   3.2960   3.2982   3.2988   
15                 time   10.0   0.1213  0.0014   3.0369   3.0373   3.0388   
7               token_3   10.0   0.0526  0.0005   2.9693   2.9695   2.9699   
17            limvfouls   10.0   0.0494  0.0003   2.9661   2.9665   2.9667   
10              period1   10.0   0.0486  0.0006   2.9652   2.9655   2.9658   
12              period3   10.0   0.0415  0.0005   2.9581   2.9584   2.9590   
16            limhfouls   10.0   0.0405  0.0003  

In [ ]:
list_results.append(result)

with open(models_folder + "20251114_results_neural_networks_all_seasons_simple_vars.pkl", "wb") as f:
    pickle.dump(list_results, f)

list_models_fitted = [item["filename_token_model"] for item in list_results]


In [31]:
list_models_fitted

['model_token_teamvars_context1_emb8_hiddenlayers1_numneurons30_multiplicador1_parameters.pth',
 'model_token_teamvars_context1_emb8_hiddenlayers1_numneurons30_multiplicador3_parameters.pth',
 'model_token_teamvars_context1_emb64_hiddenlayers1_numneurons30_multiplicador10_parameters.pth',
 'model_token_teamvars_context3_emb8_hiddenlayers5_numneurons50_multiplicador1_parameters.pth',
 'model_token_teamvars_context3_emb64_hiddenlayers1_numneurons100_multiplicador5_parameters.pth',
 'model_token_teamvars_context3_emb64_hiddenlayers3_numneurons30_multiplicador3_parameters.pth',
 'model_token_teamvars_context3_emb64_hiddenlayers5_numneurons50_multiplicador10_parameters.pth',
 'model_token_teamvars_context5_emb16_hiddenlayers3_numneurons50_multiplicador5_parameters.pth',
 'model_token_teamvars_context5_emb16_hiddenlayers3_numneurons100_multiplicador1_parameters.pth',
 'model_token_teamvars_context5_emb16_hiddenlayers3_numneurons100_multiplicador10_parameters.pth',
 'model_token_teamvars_cont